# 2.3. Benchmark Chính thức Milestone 2 trên Google Colab CPU-only

## 1. Tổng hợp 5 phiên chạy lạnh liên tiếp và đối chiếu 5 bậc dung lượng lưu trữ

Cả **5 phiên thành viên** (mỗi phiên xử lý mới `5.000` bài báo, `publications_reused = 0`, `sessions_overlap = False`) đều chạy trên **Google Colab Standard CPU-only** (`2 logical vCPU` Intel x86_64, tổng RAM `12.975,52 MiB`, Python `3.13.18`) và được kiểm chứng tự động qua `src/crawler/benchmark_sessions.py`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
tot = bench["totals"]
ep = m2["tex_elapsed_percentiles"]
df_b8 = pd.DataFrame([
    ("Môi trường xác minh tất cả 5 phiên", "Google Colab CPU-only (True) • 2 logical vCPU • 12.975,52 MiB RAM"),
    ("Kiểm tra gối giờ / trùng tệp thống kê", f"sessions_overlap = {tot['sessions_overlap']} | duplicate_skipped = {tot['duplicate_run_files_skipped']}"),
    ("Tổng số bài báo phân tích từ lạnh (reused = 0)", f"{fmt_int(tot['unique_publications'])} bài báo ({fmt_int(scope['versions'])} phiên bản)"),
    ("Tổng thời gian thực thi + kiểm chuẩn (5 phiên)", f"{fmt_num(tot['total_profiler_sec'], 2)} giây (5h 11p 32s ≈ 5,19 giờ)"),
    ("Thời gian trung bình mỗi bài báo", f"{fmt_num(tot['avg_sec_per_unique_publication'], 4)} giây/bài (0,4504 giây/phiên bản)"),
    ("Phân vị thời gian phân tích 1 phiên bản có TeX", f"p50 = {fmt_num(ep['p50'], 3)}s | p90 = {fmt_num(ep['p90'], 3)}s | p99 = {fmt_num(ep['p99'], 3)}s | max = {fmt_num(ep['max'], 2)}s"),
    ("Đỉnh bộ nhớ RAM lớn nhất quan sát được (Peak RSS)", f"{fmt_num(tot['max_session_peak_rss_mib'], 2)} MiB (< 6,6% trần RAM Colab)"),
    ("Bộ nhớ RAM trung bình gia quyền theo thời gian", f"{fmt_num(tot['session_time_weighted_avg_rss_mib'], 2)} MiB"),
    ("Đỉnh đĩa workspace tạm thời (có chỉ mục SQLite kiểm chuẩn)", f"{fmt_num(tot['max_session_peak_disk_bytes']/1024**3, 2)} GiB ({fmt_int(tot['max_session_peak_disk_bytes'])} byte)"),
    ("Dung lượng thư mục parsed/ hợp nhất toàn nhóm", f"{fmt_num(tot['final_output_bytes']/1024**3, 2)} GiB ({fmt_int(tot['final_output_bytes'])} byte)"),
], columns=["Chỉ tiêu Benchmark chính thức M2", "Giá trị xác minh"])
display(style_table(df_b8, caption="Bảng 8.1. Tổng hợp Benchmark Milestone 2 trên Google Colab CPU-only (25.000 bài báo)."))

df_sess = pd.DataFrame(bench["sessions"])
view_s2 = pd.DataFrame({
    "Phiên": df_sess["session"],
    "Số bài (cold)": df_sess["processed_attempts"],
    "Thời gian (phút)": df_sess["profiler_sec"] / 60.0,
    "TB giây/bài": df_sess["avg_sec_per_pub"],
    "Mean RSS (MiB)": df_sess["avg_rss_mib"],
    "Sampled Peak RSS (MiB)": df_sess["peak_rss_mib"],
    "OS Peak RSS (MiB)": df_sess["os_reported_peak_rss_mib"],
    "Peak Disk (GiB)": df_sess["peak_disk_bytes"] / (1024**3),
    "Parquet thuần (GiB)": df_sess["final_output_bytes"] / (1024**3),
})
display(style_table(
    view_s2,
    caption="Bảng 8.2. Chi tiết hiệu năng từng phiên Colab CPU-only của 5 thành viên (0 bài tái sử dụng).",
    int_cols=["Số bài (cold)"],
    bar_cols={"TB giây/bài": ("var(--ah-bar-fill)", 1.0)},
))

ph_m2 = pd.DataFrame([
    {
        "Pha xử lý nội bộ M2": k,
        "Số lượt gọi (calls)": int(v["calls"]),
        "Tổng thời gian (giây)": float(v["elapsed_sec"]),
        "Tỷ trọng (%)": 100.0 * float(v["elapsed_sec"]) / bench["totals"]["total_profiler_sec"],
    }
    for k, v in bench["phases_aggregated"].items()
]).sort_values("Tổng thời gian (giây)", ascending=False)
display(style_table(
    ph_m2,
    caption="Bảng 8.3. Phân rã thời gian các pha xử lý nội bộ M2 cộng dồn qua 5 phiên.",
    int_cols=["Số lượt gọi (calls)"],
    pct_cols=["Tỷ trọng (%)"],
    bar_cols={"Tổng thời gian (giây)": ("rgba(13, 148, 136, 0.22)", None)},
))

df_rec = pd.DataFrame(bench["storage_reconciliation"]).rename(columns={
    "tier": "Bậc lưu trữ & đóng gói",
    "bytes": "Dung lượng chính xác (Bytes)",
    "gib": "Quy đổi (GiB)",
    "note": "Giải trình kỹ thuật kiểm toán",
})
display(style_table(
    df_rec,
    caption="Bảng 8.4. Đối chiếu khớp tuyệt đối đến từng byte xuyên suốt 5 bậc lưu trữ từ M2 sang bộ 6 bảng Hugging Face.",
    int_cols=["Dung lượng chính xác (Bytes)"],
))

In [ ]:
df_s = pd.DataFrame(bench["sessions"])
fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.9))

# Panel 1: Tốc độ trung bình
axes[0].bar(df_s["member"], df_s["avg_sec_per_pub"], color=BOOK_PALETTE["navy"], width=0.56)
axes[0].axhline(bench["totals"]["avg_sec_per_unique_publication"], color=BOOK_PALETTE["crimson"], ls="--", lw=1.3, label="TB 0,7477s/bài")
axes[0].set_ylim(0, 0.96); axes[0].grid(axis="y"); axes[0].legend(fontsize=8.0)
axes[0].set_ylabel("Giây / bài báo"); axes[0].set_title("Hình 8.1a. Tốc độ TB mỗi bài")
for idx_i, v_i in enumerate(df_s["avg_sec_per_pub"]):
    axes[0].text(idx_i, v_i + 0.02, fmt_num(v_i, 3), ha="center", fontsize=7.8, fontweight="bold")

# Panel 2: Grouped Bar 3 cột tách biệt (x - w, x, x + w) sửa triệt để lỗi chồng cột
xb = np.arange(len(df_s)); wb = 0.26
axes[1].bar(xb - wb, df_s["avg_rss_mib"], wb, color=BOOK_PALETTE["emerald"], label="Mean RSS")
axes[1].bar(xb, df_s["peak_rss_mib"], wb, color=BOOK_PALETTE["amber"], label="Sampled Peak")
axes[1].bar(xb + wb, df_s["os_reported_peak_rss_mib"], wb, color=BOOK_PALETTE["crimson"], label="OS Peak")
axes[1].set_xticks(xb); axes[1].set_xticklabels(df_s["member"])
axes[1].set_ylim(0, 1050); axes[1].grid(axis="y"); axes[1].legend(fontsize=7.6, loc="upper right")
axes[1].set_ylabel("RAM (MiB)"); axes[1].set_title("Hình 8.1b. Bộ nhớ RAM (< 6,6% Colab)")

# Panel 3: Chu trình dung lượng đĩa mỗi phiên
axes[2].bar(xb - wb, df_s["peak_disk_bytes"] / 1024**3, wb, color=BOOK_PALETTE["slate"], label="Peak Workspace")
axes[2].bar(xb, df_s["final_disk_bytes"] / 1024**3, wb, color=BOOK_PALETTE["sky"], label="Final Workspace")
axes[2].bar(xb + wb, df_s["final_output_bytes"] / 1024**3, wb, color=BOOK_PALETTE["teal"], label="Parquet Output")
axes[2].set_xticks(xb); axes[2].set_xticklabels(df_s["member"])
axes[2].set_ylim(0, 9.6); axes[2].grid(axis="y"); axes[2].legend(fontsize=7.6, loc="upper right")
axes[2].set_ylabel("Dung lượng đĩa (GiB)"); axes[2].set_title("Hình 8.1c. Chu trình đĩa mỗi phiên")
plt.tight_layout(); plt.show()

In [ ]:
profiles = bench.get("resource_profiles") or {}
avail = {k: v for k, v in profiles.items() if v and v.get("rows")}
if avail:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 4.2))
    for label, prof in avail.items():
        f = pd.DataFrame(prof["rows"])
        t_min = f["window_end_sec"] / 60.0
        col = MEMBER_COLORS.get(label.split()[-1].strip("()"), BOOK_PALETTE["navy"])
        ax1.plot(t_min, f["rss_sample_mean_mib"], label=f"{label} (Mean)", color=col, lw=1.5)
        if "rss_peak_mib" in f.columns:
            ax1.fill_between(t_min, f["rss_sample_mean_mib"], f["rss_peak_mib"], color=col, alpha=0.12)
        disk_rows = f.dropna(subset=["disk_peak_bytes"])
        if not disk_rows.empty:
            ax2.plot(disk_rows["window_end_sec"] / 60.0, disk_rows["disk_peak_bytes"] / 1024**3, label=label, color=col, lw=1.5)
    ax1.set_xlabel("Thời gian chạy (phút)"); ax1.set_ylabel("RSS cửa sổ 60s (MiB; dải mờ = đỉnh tức thời)")
    ax1.set_title("Hình 8.2a. Diễn biến RAM phẳng (~240 MiB, 0 rò rỉ bộ nhớ)"); ax1.grid(True); ax1.legend(fontsize=7.8)
    ax2.set_xlabel("Thời gian chạy (phút)"); ax2.set_ylabel("Dung lượng workspace (GiB)")
    ax2.set_title("Hình 8.2b. Tăng đĩa tạm thời ở pha kiểm chuẩn SQLite cuối phiên"); ax2.grid(True); ax2.legend(fontsize=7.8)
    plt.tight_layout(); plt.show()

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [`src/crawler/profiler.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/profiler.py).
- Mã nguồn triển khai: [`src/crawler/benchmark_sessions.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/benchmark_sessions.py).
- Mã nguồn triển khai: [`tools/m2_colab_runner.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).